# Underwater vision-enhanced segmentation: train and evaluate on Google Colab

Runtime → Change runtime type → **GPU**. Depth maps are computed automatically with Depth Anything the first time each image is used.

In [ ]:
!git clone https://github.com/Saeed-Talamkhani/underwater-vision-segmentation.git
%cd underwater-vision-segmentation
!pip install -q -r requirements.txt

## Data
Mount Google Drive and point `DATA_ROOT` to a folder with `train/`, `val/` and `test/`, each containing `images/` and `masks/` (see `data/README.md`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/path/to/data'  # <- change this

## Train
Same settings as the paper (`configs/default.yaml`): 60 epochs, batch 16, Adam 1e-4, focal loss, early stopping after 20 epochs.

In [ ]:
!python train.py --config configs/default.yaml --data-root "$DATA_ROOT" --output-dir runs/unet5c

## Evaluate

In [ ]:
!python evaluate.py --checkpoint runs/unet5c/best.pth --data-root "$DATA_ROOT" --split test \
    --per-image-csv runs/unet5c/iou_per_image_test.csv --save-dir runs/unet5c/predictions_test

## Predict on new images
No masks or depth maps are needed.

In [ ]:
!python predict.py --checkpoint runs/unet5c/best.pth --input "$DATA_ROOT/test/images" --output outputs/

## Save the weights to Drive
`export_weights.py` removes the optimizer state, which makes the file about 3x smaller.

In [ ]:
!python export_weights.py --checkpoint runs/unet5c/best.pth --out weights/unet5c.pth
!cp weights/unet5c.pth /content/drive/MyDrive/